# Travel Personas
- using k means to try and find the type of traveller i am on different days

### Imports

In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import plotly.express as px
import plotly.graph_objects as go
import datetime

import json

from transformers import pipeline

/Users/sunnywillert/GitHub/floating-in-space/py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Variables

In [2]:
hours_in_a_day = 24

In [3]:
regions = {
    'South America': ['Colombia','Ecuador','Peru','Bolivia','Brasil'],
    'Central America': ['Guatemala','Honduras','Nicaragua','El Salvador'],
    'Southern Africa': ['South Africa','Malawi','Mozambique','Eswatini','Lesotho']
}

In [4]:
base_currency = ['CAD']

In [5]:
countries = {
    'Colombia' : {
        'color': '#EFCA08',
        'currency': ['COP']
    },
    'Ecuador' : {
        'color': '#FF9000',
        'currency': ['USD']
    },
    'Peru' : {
        'color': '#DB5461',
        'currency': ['PEN']
    },
    'Bolivia' : {
        'color': '#A4243B',
        'currency': ['BOB']
    },
    'Brasil' : {
        'color': '#94BFA7',
        'currency': ['BRL']
    },
    'Guatemala' : {
        'color': '#86A5D9',
        'currency': ['GTQ','USD']
    },
    'Honduras' : {
        'color': '#0B3954',
        'currency': ['HNL','USD']
    },
    'Nicaragua' : {
        'color': '#57C4E5',
        'currency': ['NIO','USD']
    },
    'El Salvador': {
        'color': '#273043',
        'currency': ['USD']
    },
    'South Africa': {
        'color': '#D4E09B',
        'currency': ['ZAR']
    },
    'Malawi': {
        'color': '#D72638',
        'currency': ['MK','BMK']
    },
    'Mozambique': {
        'color': '#2D5C1E',
        'currency': ['MZN']
    },
    'Eswatini': {
        'color': '#AFA3B8',
        'currency': ['ZAR']
    },
    'Lesotho': {
        'color': '#8EF9F3',
        'currency': ['ZAR']
    }
}

In [6]:
transport_types = {
    'hitchhiking': {
        'values': ['hitchhiking'],
        'mapping': 0
    },
    'self': {
        'values': ['car', 'car share'],
        'mapping': 1
    },
    'bus': {
        'values': ['bus'],
        'mapping': 2
    },
    'tourist': {
        'values': ['van', 'truck', 'flight', 'teleferico', 'jeep', 'boat', 'shuttle'],
        'mapping': 3
    },
    'local': {
        'values': ['chicken bus', 'mini bus', 'chapa', 'kombi', 'collectivo', 'train'],
        'mapping': 4
    },
    'multiple': {
        'values': [],
        'mapping': 5
    }
}

### Functions

In [7]:
def map_countries(df, timeline):
    for idx, row in timeline.iterrows():
        # get all rows that occur in the dates within a country timeline and see if the row has the currency of a country
        clip = df[(df.date >= row.entry_date) & (df.date <= row.exit_date) & (df.currency.isin(countries[row.country]['currency'] + base_currency))]
    
        df.loc[clip.index.values, 'country'] = row.country

    return df

In [8]:
# formatting columns to be nicer to work with
def format_names(df):
    for col in df.columns:
        df = df.rename(columns={col: col.replace(' ','_')})

    return df

### Base data

In [9]:
timeline = pd.read_csv('../data/country_timeline.csv')
timeline['entry_date'] = pd.to_datetime(timeline['entry_date'])#, format='%m/%d/%Y')
timeline['exit_date'] = pd.to_datetime(timeline['exit_date'])#, format='%m/%d/%Y')

In [10]:
money = pd.read_csv('../data/money_manager_2024_2025.csv')
money.columns = money.columns.str.lower()
money = money.rename(columns={'income/expense':'income_expense',' ':'date'})

In [11]:
money['date'] = pd.to_datetime(money['date'], format='%m/%d/%Y %H:%M:%S').dt.normalize()
money['month'] = money['date'].dt.strftime("%B %Y")

In [12]:
money = map_countries(money, timeline)

In [13]:
# fix two lines in money data set that need a different category
change_cat = ['Towel','Volunteer fee at the pink iguana']
money.loc[money.note.isin(change_cat), 'category'] = 'Other'

In [14]:
transport = pd.read_csv('../data/transport_data.csv')
transport = format_names(transport)
transport['date'] = pd.to_datetime(transport['date'], format='%m/%d/%Y')

# have to get rid of the random extra column i added for some reason
transport = transport.drop('Unnamed:_8', axis=1)

# fixing hitch hiking split
transport['type'] = transport['type'].str.replace('hitch hiking','hitchhiking')
transport['type'] = transport['type'].str.replace('bus ','bus')

## Making the feature vectors
- 1 day into a bunch of features
- spending:
  - ~purchases sum for each category
  - ~number of purchases for each category
  -  maybe have the average price per purchase per category?
  - ~sum of dollars spent
  - ~allowance
  - ~diff between allowance and spending
  - ~number of time purchases made
- transport:
  - ~hours spent travelling
  - ~number of different transports taken
  - ~transport type label
- activities/tours:
  - activity done (will need to map this out to different days because activities are put in by date & length of time)
  - tour done (will need to map this out to different days because activities are put in by date & length of time)
  - label of activity type
  - label of tour done
  - time spent on activity
  - time spent on tour
- extra randoms
  - split type of accomodations into values: dorm, camping, suite, unpaid
- location data:
  - ~~country numerical value - dont think i want to bias by country~~

#### Spending Features

In [15]:
# grouping by to get dollar sum and number of times sum
money_agg = money.groupby(['country','month','date','income_expense']).agg({'cad':'sum', 'currency':'count'}).reset_index()

# pivot the table to get the categories as 
mpiv = money_agg.pivot(index=['country','month','date'], columns='income_expense', values=['cad','currency']).reset_index()
mpiv.columns = [col[0] if col[0] in ['country','month','date'] else'_'.join(map(str, col)).strip() for col in mpiv.columns.values]

# rename columns and keep only necessary data
mpiv = mpiv.rename(columns={
    'cad_Expense':'total_expense',
    'cad_Income': 'budget',
    'currency_Expense': 'total_num_purchases'
}).drop('currency_Income', axis=1)

# get budget/spending difference
mpiv['difference'] = mpiv.total_expense - mpiv.budget

In [16]:
mpiv

,country,month,date,total_expense,budget,total_num_purchases,difference
0,Bolivia,August 2024,2024-08-01,108.31,60.0,4.0,48.31
1,Bolivia,August 2024,2024-08-02,51.68,60.0,8.0,-8.32
2,Bolivia,August 2024,2024-08-03,63.72,60.0,8.0,3.72
3,Bolivia,August 2024,2024-08-04,49.72,60.0,8.0,-10.28
4,Bolivia,August 2024,2024-08-05,85.46,60.0,4.0,25.46
...,...,...,...,...,...,...,...
424,South Africa,September 2025,2025-09-21,25.63,60.0,2.0,-34.37
425,South Africa,September 2025,2025-09-22,53.95,60.0,4.0,-6.05
426,South Africa,September 2025,2025-09-23,511.07,60.0,4.0,451.07
427,South Africa,September 2025,2025-09-24,19.32,60.0,1.0,-40.68


In [17]:
# do some data simplification
money_copy = money.copy()

# simplify the category strings
money_copy['category'] = money_copy['category'].str.split(' ').str[-1].str.lower()

# some categories are very small and can be compressed
# transport = transport + flight
# health_beauty = health, beauty and tattoo
# other = other, household, education, crafting
money_copy.loc[money_copy.category == 'flight','category'] = 'transport'
money_copy.loc[money_copy.category.isin(['other','household','education','crafting']),'category'] = 'other'
money_copy.loc[money_copy.category.isin(['health','beauty','tattoo']),'category'] = 'other'

In [18]:
# now to get category breakdown data
mcat = money_copy[money_copy.income_expense == 'Expense'].groupby(['country','month','date','category']).agg({'cad':'sum', 'currency':'count'}).reset_index()

# fix columns names before pivot
mcat = mcat.rename(columns={'cad':'expense', 'currency':'num_purchases'})

# pivot data out into categories
mcat = mcat.pivot(index=['country','month','date'], columns='category', values=['expense','num_purchases']).reset_index()
mcat.columns = [col[0] if col[0] in ['country','month','date'] else'_'.join(map(str, col)).strip() for col in mcat.columns.values]

# fill nans
mcat = mcat.fillna(0).rename(columns={'expense_life':'expense_social_life','num_purchases_life':'num_purchases_social_life'})

In [19]:
# bring together the spending features
f_money = pd.merge(mpiv, mcat, how='inner', on=['country','month','date'])
f_money.head()

,country,month,date,total_expense,budget,total_num_purchases,difference,expense_accommodation,expense_adventure,expense_apparel,...,expense_transport,num_purchases_accommodation,num_purchases_adventure,num_purchases_apparel,num_purchases_culture,num_purchases_food,num_purchases_gift,num_purchases_social_life,num_purchases_other,num_purchases_transport
0,Bolivia,August 2024,2024-08-01,108.31,60.0,4.0,48.31,16.77,0.00,0.00,...,73.78,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
1,Bolivia,August 2024,2024-08-02,51.68,60.0,8.0,-8.32,13.02,2.96,0.00,...,1.97,1.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,1.0
2,Bolivia,August 2024,2024-08-03,63.72,60.0,8.0,3.72,13.02,0.00,3.95,...,0.00,1.0,0.0,1.0,1.0,3.0,0.0,1.0,1.0,0.0
3,Bolivia,August 2024,2024-08-04,49.72,60.0,8.0,-10.28,0.00,0.00,0.00,...,20.72,0.0,0.0,0.0,0.0,4.0,0.0,0.0,2.0,2.0
4,Bolivia,August 2024,2024-08-05,85.46,60.0,4.0,25.46,0.00,72.34,0.00,...,0.00,0.0,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0


#### Transport Features
- transport type: made the category more basic, but sometimes there is more than 1 type of travel... maybe add a final group that is mixed?

In [20]:
# invert the dictionary
type_to_category = {
    v: category 
    for category, info in transport_types.items() 
    for v in info['values']
}

# apply mapping to dataframe
transport['category'] = transport['type'].map(type_to_category)

In [21]:
# get all of the transport categories that have happened in one day
f_transport = transport.groupby(['country','date']).agg({'approximate_time':'sum', 'type':'count', 'category':lambda x: list(set(x))}).reset_index()

# simplify the category: if multiple, replace with value 'multiple'
f_transport['category'] = f_transport['category'].apply(lambda x: x[0] if len(x) == 1 else 'multiple')

In [22]:
# extract the mapping
category_id_map = {k: v['mapping'] for k, v in transport_types.items()}

# assign the mapping
f_transport['category_id'] = f_transport['category'].map(category_id_map)

In [23]:
f_transport.head()

,country,date,approximate_time,type,category,category_id
0,Bolivia,2024-07-02,0.5,1,bus,2
1,Bolivia,2024-07-03,4.0,1,tourist,3
2,Bolivia,2024-07-06,6.5,2,multiple,5
3,Bolivia,2024-07-08,3.0,2,tourist,3
4,Bolivia,2024-07-10,13.0,1,bus,2


#### Activities/Tours Features

#### Other Random Features

In [24]:
money[money.category == '🏠 Accommodation']

,date,account,category,subcategory,note,cad,income_expense,description,amount,currency,account.1,month,country
36,2025-09-16,Card,🏠 Accommodation,NaN,Forever resort campsite,12.63,Expense,NaN,170.0,ZAR,12.63,September 2025,South Africa
42,2025-09-15,Card,🏠 Accommodation,NaN,Semonkong campsite,13.38,Expense,NaN,180.0,ZAR,13.38,September 2025,Lesotho
44,2025-09-14,Card,🏠 Accommodation,NaN,Semonkong campsite,13.38,Expense,NaN,180.0,ZAR,13.38,September 2025,Lesotho
52,2025-09-13,Card,🏠 Accommodation,NaN,Sani pass dorm,11.15,Expense,NaN,150.0,ZAR,11.15,September 2025,Lesotho
56,2025-09-12,Cash,🏠 Accommodation,NaN,Mamohase Dorm,11.15,Expense,NaN,150.0,ZAR,11.15,September 2025,Lesotho
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2729,2024-07-06,Cash,🏠 Accommodation,NaN,La Paz hostel,10.85,Expense,NaN,55.0,BOB,10.85,July 2024,Bolivia
2736,2024-07-05,Cash,🏠 Accommodation,NaN,Hostel mirador,7.89,Expense,NaN,40.0,BOB,7.89,July 2024,Bolivia
2741,2024-07-04,Cash,🏠 Accommodation,NaN,Hostel mirador,7.89,Expense,NaN,40.0,BOB,7.89,July 2024,Bolivia
2748,2024-07-03,Cash,🏠 Accommodation,NaN,Hostel mirador,7.89,Expense,NaN,40.0,BOB,7.89,July 2024,Bolivia


In [25]:
# unpaid accomodation is anytime we have a day that has an allowance, but has no accomodation expense
# get the days that allowance is paid out (travel days) and accom days
# find days without accomodation -- unpaid
travel_days = money[(money.category == '🤑 Allowance') & (money.note.isna())][['date']]
accom_days = money[money.category == '🏠 Accommodation'][['date','note','cad']]

accom = pd.merge(travel_days, accom_days, how='left', on='date')

# label the unpaid accom days
accom.loc[accom.note.isna() & accom.cad.isna(), 'accomodation_type'] = 'unpaid'

### Zero Shot Classifier
- identifying type of accomodation: dorm, camping, suite, unpaid

In [26]:
# Load the classifier (this will download a pre-trained model)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

In [27]:
candidate_labels = ['dorm', 'camping', 'suite']

In [29]:
paid_only = accom[accom.accomodation_type != 'unpaid']

In [30]:
def classify_travel(text):
    result = classifier(text, candidate_labels)
    # The first label in 'labels' is the one with the highest score
    return result['labels'][0]

In [31]:
paid_only['category'] = paid_only['note'].apply(classify_travel)

can i take the correct labels that we have, then use them to train a model and predict the incorrect labels?

In [37]:
incorrectly_labelled = [
    'Underwater explorer safari tent','Crown lodge single',
    'Blantyre baobab backpackers','B hive room','Coral diving tent','Glamping dorm st lucia','Pelícano surf camp',
    'Wabi sabi','Solid surf','Hostel hammock','Hostel azul','Playa de la fuente','San sebastian antigua','Iguana perdida',
    'Hostal vista la mar','Aquarela do leme','Long board paradise','Private room share with beni','Second itacaré hostel',
    'Personal hostel room','Surfcamp arara','One bed in morro branco','Airbnb hostel','Tent','Selina bonito','Villa oro'
]

In [41]:
paid_only[paid_only.note.isin(incorrectly_labelled)]

,date,note,cad,accomodation_type,category
24,2025-09-01,Underwater explorer safari tent,12.93,NaN,suite
25,2025-08-31,Underwater explorer safari tent,12.93,NaN,suite
59,2025-07-29,Crown lodge single,3.95,NaN,dorm
74,2025-07-14,Blantyre baobab backpackers,15.80,NaN,camping
92,2025-06-27,B hive room,22.29,NaN,dorm
98,2025-06-21,Coral diving tent,23.41,NaN,suite
99,2025-06-20,Coral diving tent,23.41,NaN,suite
100,2025-06-19,Coral diving tent,23.41,NaN,suite
101,2025-06-18,Coral diving tent,23.41,NaN,suite
107,2025-06-12,Glamping dorm st lucia,20.06,NaN,camping


In [33]:
pd.set_option('display.max_rows', None)